# **Tutorial for Querying Bluesky Data with ClickHouse**

This notebook will explain how to connect to the Diderot Bluesky ClickHouse database and querying each ingested API table. From Chengyi's understanding, this is a mirror of the Bluesky content from mid August 2026.

**Prerequisites**

* Conda environment from ``environment.yml (diderot-clinic)``
* A ``.env`` file in the repo root with ``CLICKHOUSE_USERNAME`` and ``CLICKHOUSE_PASSWORD`` provided by Henri

## **Table of Contents**
| Section | Name | Description |
|---------|------|-------------|
| 0 | [Connect and Inspect](#0-connect-and-inspect) | Connect to database and inspect data. |
| 1 | [Posts](#1-posts-records_app_bsky_feed_post) | Post records |
| 2 | [Reposts](#2-posts-records_app_bsky_feed_repost) | Repost records |
| 3 | [Likes](#3-likes-records_app_bsky_feed_like) | Like records |
| 4 | [Follows](#4-follows-records_app_bsky_graph_follow) | Follow records |
| 5 | [Profiles](#5-profiles-records_app_bsky_actor_profile) | Profile records |
| 6 | [Glossary](#6-glossary) | Glossary of all API endpoints |

## 0. Connect and Inspect

We first connect to the database through a clickhouse client.

In [1]:
import clickhouse_connect # clickhouse API
import os # environment variables
import pandas as pd # for data visualization
from dotenv import load_dotenv # load .env file

load_dotenv()

client = clickhouse_connect.get_client(
    host="ch.bsky.diderot.app",
    port=8443,
    username=os.getenv("CLICKHOUSE_USERNAME"),
    password=os.getenv("CLICKHOUSE_PASSWORD"),
    secure=True,
)


def run(sql: str):
    """Run a query and return (column_names, rows)."""
    result = client.query(sql)
    return result.column_names, result.result_rows


def show(sql: str, max_rows: int = 10000):
    """Run a query and display the result as a pandas DataFrame."""
    cols, rows = run(sql)
    df = pd.DataFrame(rows, columns=cols)
    display(df.head(max_rows))

Now we can view the tables that live in the ``bluesky_ingest`` database provided by Diderot.

In [2]:
show("SHOW TABLES FROM bluesky_ingest")

,name
0,records
1,records_app_bsky_actor_profile
2,records_app_bsky_actor_status
3,records_app_bsky_feed_generator
4,records_app_bsky_feed_generator__description_f...
5,records_app_bsky_feed_like
6,records_app_bsky_feed_post
7,records_app_bsky_feed_post__entities
8,records_app_bsky_feed_post__facets
9,records_app_bsky_feed_postgate


## 1. Posts (`records_app_bsky_feed_post`)

Approx. size: ~2.8B rows (please use `LIMIT`s and filters; avoid unbounded `GROUP BY` or full-table scans)

This table contains ingested Bluesky post records (creates, updates, and deletes). Each row is one firehose event for an app.bsky.feed.post record.

### 1.1 Inspect the schema

You could run the snippet below to see the columns in the ingested post records.

In [3]:
# show("DESCRIBE bluesky_ingest.records_app_bsky_feed_post")

Alternatively, here is the summary:

| Column | Type | Description |
| ------ | ---- | ----------- |
| `tid_us` | `UInt64` | Ingest timestamp (microseconds) |
| `did` | `String` | Author DID |
| `rkey` | `String` | Record key (post id within the repo) |
| `cid` | `String` | Content ID of the record |
| `action` | `Enum8('create','update','delete')` | Firehose action |
| `rev` | `String` | Repo revision |
| `live` | `Bool` | Whether the record is currently live |
| `text` | `Nullable(String)` | Post text |
| `created_at` | `Nullable(String)` | Author-declared creation time (ISO-8601) |
| `embed_type` | `LowCardinality(Nullable(String))` | Embed lexicon type, e.g. `app.bsky.embed.images` |
| `embed_record_uri` / `embed_record_cid` | `Nullable(String)` | Quote / record embed |
| `embed_external_*` | various | Link card embed fields |
| `embed_video_*` | various | Video embed fields |
| `embed_image_*` | arrays | Image CIDs, alts, aspect ratios |
| `labels` | `Array(String)` | Self-labels on the post |
| `langs` | `Array(String)` | Declared language codes |
| `reply_root_uri` / `reply_root_cid` | `Nullable(String)` | Root of the thread (if reply) |
| `reply_parent_uri` / `reply_parent_cid` | `Nullable(String)` | Parent post (if reply) |
| `tags` | `Array(String)` | Hashtag-style tags |
| `event_time` | `DateTime64(6)` | Event time in ClickHouse |

### 1.2 First Query

With the snippet below, you can make your first databse query to show posts! Useful columns to start with would be `did` (author ID), `rkey` (record key, i.e. post ID), `text` (the text of the post itself), `created_at` (post creation time), `action` (what was being done to the post), `live` (whether the record is currently live, i.e. live at the time of the mirror), and `embed_type` (embeded media such as images/videos/audio etc.).

In [4]:
show(
    """
    SELECT did, rkey, action, live, text, created_at, embed_type, langs, tags
    FROM bluesky_ingest.records_app_bsky_feed_post
    LIMIT 5
    """
)

,did,rkey,action,live,text,created_at,embed_type,langs,tags
0,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jj4ws2h,create,False,a,2024-07-17T13:52:47.000Z,NaN,[],[]
1,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jj4wtjo,create,False,b,2024-07-17T13:52:47.001Z,NaN,[],[]
2,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb3267,create,False,c,2024-07-17T13:53:24.000Z,NaN,[],[]
3,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb33on,create,False,d,2024-07-17T13:53:24.001Z,NaN,[],[]
4,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantkoraasfx,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:04:08.1736565+00:00,app.bsky.embed.images,[],[]


> **IMPORTANT**: Remember to always limit your queries to prevent timeout!

We have a maximum query time of 120s.

### 1.3 Sampling posts with text

As the above query results suggest, some posts have been deleted or have useless information. To filter for useful posts, we may add a WHERE field in the SQL:

In [5]:
show(
    """
    SELECT did, rkey, action, live, text, created_at, embed_type, langs, tags
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE text IS NOT NULL AND length(text) > 10
    LIMIT 5
    """
)

,did,rkey,action,live,text,created_at,embed_type,langs,tags
0,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantkoraasfx,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:04:08.1736565+00:00,app.bsky.embed.images,[],[]
1,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantoevwdxu5,create,False,hi from PinkSea's bsky integration!,2024-11-11T09:06:12.0594507+00:00,app.bsky.embed.images,[],[]
2,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranu625xochs,create,False,https://localhost:5173/did:plc:2edipcwcjiezjta...,2024-11-11T09:14:57.6594223+00:00,app.bsky.embed.images,[],[]
3,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvf5cjjqgr,create,False,https://localhost:5173/did:plc:2edipcwcjiezjta...,2024-11-11T09:36:49.5745663+00:00,app.bsky.embed.images,[],[]
4,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvlk5udvbp,create,False,https://example.com/did:plc:2edipcwcjiezjtanjs...,2024-11-11T09:40:24.3800382+00:00,app.bsky.embed.images,[],[]


### 1.4 Posts by one author (`did`)

Each user has a unique `did`. To obtain a user's `did`, you may run the following function:

In [6]:
import urllib.request, json
def get_did(handle: str) -> str:
    """Get the DID of a given handle."""  
    url = f"https://public.api.bsky.app/xrpc/com.atproto.identity.resolveHandle?handle={handle}"
    return json.load(urllib.request.urlopen(url))["did"]

For example, we can run this on Alexandria Ocasio-Cortez's bluesky handle:

In [7]:
aoc_handle = "aoc.bsky.social"
aoc_did = get_did(aoc_handle)
print(aoc_did)

did:plc:p7gxyfr5vii5ntpwo7f6dhe2


We can then filter posts made by her:

In [8]:
show(
    f"""
    SELECT did, rkey, text, created_at, action, live
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE did = '{aoc_did}'
    LIMIT 5
    """
)

,did,rkey,text,created_at,action,live
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juexmwyc3o2h,Hi there! Is this thing on? 🎤,2023-04-27T20:42:40.540Z,create,False
1,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf44f52li2k,Ok! Here is Deco at the airport from a few wee...,2023-04-27T22:02:53.788Z,create,False
2,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf52klwc72i,Here you go!,2023-04-27T22:19:46.107Z,create,False
3,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf56kh5ad2f,This is so sweet 🥹 thank you,2023-04-27T22:22:00.161Z,create,False
4,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,3juf5lltngs2j,I just got here and it’s already more enjoyabl...,2023-04-27T22:29:17.738Z,create,False


### 1.5 Replies only

Replies set `reply_parent_uri` (and usually `reply_root_uri`). URIs look like:
`at://did:plc:.../app.bsky.feed.post/<rkey>`.

In [9]:
show(
    """
    SELECT did, rkey, text, created_at, reply_root_uri, reply_parent_uri
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE reply_parent_uri IS NOT NULL AND text IS NOT NULL
    LIMIT 5
    """
)

,did,rkey,text,created_at,reply_root_uri,reply_parent_uri
0,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jj4wtjo,b,2024-07-17T13:52:47.001Z,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...
1,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb3267,c,2024-07-17T13:53:24.000Z,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...
2,did:plc:24chxmo2sw7ul7mzvvbj6fwh,223m52jkb33on,d,2024-07-17T13:53:24.001Z,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...,at://did:plc:24chxmo2sw7ul7mzvvbj6fwh/app.bsky...
3,did:plc:2226xwprhlf5q3z7lzonrkuv,3jqbrcnqdsk2u,What are the valuations for payment companies?...,2023-03-06T16:30:47.209Z,at://did:plc:qjeavhlw222ppsre4rscd3n2/app.bsky...,at://did:plc:qjeavhlw222ppsre4rscd3n2/app.bsky...
4,did:plc:23rzzv4qyw54qsu6lqqham2o,3jqc7k5edb22e,rindo com o “dar um raio”,2023-03-06T20:45:30.356Z,at://did:plc:p52irwmwli3mlkfy3c33mt4b/app.bsky...,at://did:plc:p52irwmwli3mlkfy3c33mt4b/app.bsky...


### 1.6 Posts with embeds

`embed_type` is a Bluesky embed lexicon, e.g. `app.bsky.embed.images` or `app.bsky.embed.external`.

In [10]:
show(
    """
    SELECT did, rkey, text, embed_type, embed_external_uri, embed_external_title,
           length(embed_image_cids) AS n_images
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE embed_type IS NOT NULL
    LIMIT 5
    """
)

,did,rkey,text,embed_type,embed_external_uri,embed_external_title,n_images
0,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantkoraasfx,hi from PinkSea's bsky integration!,app.bsky.embed.images,None,None,1
1,did:plc:2edipcwcjiezjtanjs5vmrlw,2rantoevwdxu5,hi from PinkSea's bsky integration!,app.bsky.embed.images,None,None,1
2,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranu625xochs,https://localhost:5173/did:plc:2edipcwcjiezjta...,app.bsky.embed.images,None,None,1
3,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvf5cjjqgr,https://localhost:5173/did:plc:2edipcwcjiezjta...,app.bsky.embed.images,None,None,1
4,did:plc:2edipcwcjiezjtanjs5vmrlw,2ranvlk5udvbp,https://example.com/did:plc:2edipcwcjiezjtanjs...,app.bsky.embed.images,None,None,1


### 1.7 Simple text search

`ILIKE` works, but can be slow on the full table. Keep `LIMIT` small to avoid timeout.

In [12]:
show(
    """
    SELECT did, rkey, text, created_at
    FROM bluesky_ingest.records_app_bsky_feed_post
    WHERE text ILIKE '%meow%'
    LIMIT 10
    """
)

,did,rkey,text,created_at
0,did:plc:ik2tenugs6vlfbj4pmepzqpp,3ju52yfo5ia2a,Meow world,2023-04-24T17:21:28.421Z
1,did:plc:g22wrkiovld7mymdemad34t3,3jurbbvlqdx2v,Meow,2023-05-02T18:07:24.432Z
2,did:plc:c54vmaxhquiocmettp5zgvka,3jrmrxrlvz22i,Meowchelin star chef,2023-03-23T19:07:04.640Z
3,did:plc:mou6t7mhh4ddhcake2a4d53r,3jty2hpuelt2u,なるほど Meow安寺というのもあったかニャ๑(ΦωΦ)๑,2023-04-22T17:28:49.768Z
4,did:plc:244dhc5kw7asrhd7i4li7khx,3jt3xf3qzcd22,New and exciting homeowner bit where the tempe...,2023-04-11T13:19:07.917Z
5,did:plc:246htihk27sb6zbixesib7cz,3jue32gxxgd2n,gmeow,2023-04-27T12:11:15.006Z
6,did:plc:pwf67cusdbxrptb4d463rbj7,3jukmolaasw2s,does anyone want to give a beautiful princess ...,2023-04-30T02:42:42.628Z
7,did:plc:p62i4mmnrxwkc7j7i5lwubcq,3jukdh5cdpe2r,Meow.,2023-04-29T23:57:29.572Z
8,did:plc:uzcybrp24kbuyukq6cldabyj,3juueyadp3d2e,"a more realistic valuation, assuming the homeo...",2023-05-03T23:51:34.323Z
9,did:plc:6ar2njsramxjgfljrvr5bjr4,3jx7ehhjw2k2i,"My cat’s name has devolved from Chairman Meow,...",2023-06-02T19:31:51.703Z


## 2. Reposts (`records_app_bsky_feed_repost`)

## 3. Likes (`records_app_bsky_feed_like`)

## 4. Follows (`records_app_bsky_graph_follow`)

## 5. Profiles (`records_app_bsky_actor_profile`)

## 6. Glossary